# Phase 1 -- Data Audit

Alert Intelligence Engine -- master plan Phase 1 (`docs/Alert_Intelligence_ML_PoC_AI_Agent_Development_Master_Plan.docx`, section 20).

**Objective:** produce a validated schema, data-quality findings, duplicate/leakage registry and outcome-label audit for `Alerts_Samples.xlsx`, before any feature engineering or model training begins.

**Non-negotiable rules in force for this phase** (master plan section 1):
- Do not manufacture ground-truth labels from `Released`/`UPS`/`Followup`.
- Do not silently remove, repair, impute, or transform data -- this notebook only *observes* and *reports*.
- Every finding here must be reproducible from code, not asserted from memory.

This notebook is read-only against the source data. No file under `data/validated|normalized|processed/` is written here -- that begins in Phase 2.

In [1]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO_ROOT))

import json
import pandas as pd

from pipelines.ingestion.load_alerts import load_raw_alerts, get_raw_data_path, SHEET_NAMES

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 160)

RAW_PATH = get_raw_data_path()
print("Raw data path:", RAW_PATH)
print("Exists:", RAW_PATH.exists())

Raw data path: /home/chpl/Documents/AI-validation/AI_validator/Alert-AI/data/raw/Alerts_Samples.xlsx
Exists: True


## 1. Schema

In [2]:
sheets = load_raw_alerts()

for name, df in sheets.items():
    print(f"=== {name} ===")
    print(f"rows={len(df)}, cols={len(df.columns)}")
    print(df.dtypes)
    print()

=== CustomerViolation ===
rows=2397, cols=23
UIN                                             int64
Customer Type                                  object
Trxn Type                                     float64
Alerted Party Name                             object
Alerted Party DOB                      datetime64[ns]
Alerted Party POB                             float64
Alerted Party Nationality                      object
Hit Details (Name)                             object
Hit Details (DOB)                              object
Hit Details (POB)                             float64
Hit Details (Nationality)                      object
Hit Details (Aditional Information)           float64
Matched Screening %                           float64
Sanctions Screening List Name                  object
Alerted Party                                  object
Branch Name                                    object
Alert Generated Date & Time            datetime64[ns]
Alert Type                           

## 2. Row/column counts vs. audit spec

Cross-check against the counts recorded in `docs/Alert_Intelligence_Unsupervised_ML_Data_Science_Feasibility_Report.docx` section 3.

In [3]:
EXPECTED = {
    "CustomerViolation": (2397, 23),
    "TransactionNameViolation": (2000, 25),
    "Rule": (2244, 24),
}

schema_check = {}
for name, (exp_rows, exp_cols) in EXPECTED.items():
    df = sheets[name]
    ok = (len(df), len(df.columns)) == (exp_rows, exp_cols)
    schema_check[name] = {
        "rows": len(df), "cols": len(df.columns),
        "expected_rows": exp_rows, "expected_cols": exp_cols,
        "match": ok,
    }
    print(f"{name}: rows={len(df)} (expected {exp_rows}), cols={len(df.columns)} (expected {exp_cols}) -> {'PASS' if ok else 'FAIL'}")

assert all(v["match"] for v in schema_check.values()), "Schema mismatch vs audit spec -- stop and investigate"
print("\nAll sheets match the audited spec.")

CustomerViolation: rows=2397 (expected 2397), cols=23 (expected 23) -> PASS
TransactionNameViolation: rows=2000 (expected 2000), cols=25 (expected 25) -> PASS
Rule: rows=2244 (expected 2244), cols=24 (expected 24) -> PASS

All sheets match the audited spec.


## 3. Duplicate and repeated-entity audit

Exact duplicate rows must be identified before any train/eval split -- letting them through would inflate evaluation and cause leakage (master plan section 3, "Rule" data-facts table).

In [4]:
dup_report = {}
for name, df in sheets.items():
    n_dupe_rows = int(df.duplicated(keep="first").sum())
    dup_report[name] = n_dupe_rows
    print(f"{name}: {n_dupe_rows} exact duplicate rows (beyond first occurrence)")

CustomerViolation: 8 exact duplicate rows (beyond first occurrence)
TransactionNameViolation: 537 exact duplicate rows (beyond first occurrence)
Rule: 0 exact duplicate rows (beyond first occurrence)


In [5]:
id_cols = {
    "CustomerViolation": ["UIN"],
    "TransactionNameViolation": ["UIN", "Trxn Ref Number"],
    "Rule": ["Customer Number", "Reference Number"],
}

id_report = {}
for name, cols in id_cols.items():
    df = sheets[name]
    report = {c: int(df[c].nunique(dropna=True)) for c in cols}
    id_report[name] = report
    print(f"{name}: " + ", ".join(f"unique {c}={n}" for c, n in report.items()))

CustomerViolation: unique UIN=1232
TransactionNameViolation: unique UIN=373, unique Trxn Ref Number=402
Rule: unique Customer Number=1106, unique Reference Number=1619


## 4. Missingness

Pandas' default `na_values` (includes the literal string `"NULL"` used throughout this workbook) is relied on -- confirmed against a manual, independent audit to produce identical counts (e.g. `Alerted Party DOB` in `CustomerViolation`: 44/2397 = 1.8% missing via the literal `"NULL"` token, not a blank cell).

In [6]:
missingness = {}
for name, df in sheets.items():
    miss_pct = (df.isna().mean() * 100).round(1).sort_values(ascending=False)
    missingness[name] = miss_pct.to_dict()
    print(f"=== {name}: fields >0% missing ===")
    print(miss_pct[miss_pct > 0])
    print()

=== CustomerViolation: fields >0% missing ===
Hit Details (Aditional Information)    100.0
Trxn Type                              100.0
Alerted Party POB                      100.0
Hit Details (POB)                      100.0
Hit Details (Nationality)               64.4
Hit Details (DOB)                       30.2
Maker Comment                            8.1
Maker Comment Date                       8.1
Maker Name                               8.1
Alerted Party DOB                        1.8
dtype: float64

=== TransactionNameViolation: fields >0% missing ===
Hit Details (POB)                      100.0
Hit Details (Aditional Information)    100.0
Alerted Party POB                      100.0
Alerted Party DOB                       78.8
Hit Details (DOB)                       62.2
Hit Details (Nationality)                5.9
Alerted Party Nationality                4.2
Maker Comment Date                       1.5
Maker Name                               1.5
Maker Comment                 

In [7]:
# Fields the master plan explicitly calls out -- verify the specific numbers, not just skim the table
checks = [
    ("CustomerViolation", "Alerted Party DOB", 1.8),
    ("CustomerViolation", "Alerted Party POB", 100.0),
    ("TransactionNameViolation", "Alerted Party DOB", 78.8),
    ("TransactionNameViolation", "Hit Details (DOB)", 62.2),
    ("TransactionNameViolation", "Alerted Party POB", 100.0),
    ("Rule", "Beneficiary Id Number", 97.1),
    ("Rule", "Beneficiary Name", 22.9),
    ("Rule", "Currency Name", 16.9),
    ("Rule", "Purpose", 14.8),
]

for sheet, col, expected_pct in checks:
    actual = round(sheets[sheet][col].isna().mean() * 100, 1)
    status = "PASS" if abs(actual - expected_pct) < 0.15 else "FAIL"
    print(f"{status}: {sheet}.{col} missing={actual}% (spec={expected_pct}%)")

PASS: CustomerViolation.Alerted Party DOB missing=1.8% (spec=1.8%)
PASS: CustomerViolation.Alerted Party POB missing=100.0% (spec=100.0%)
PASS: TransactionNameViolation.Alerted Party DOB missing=78.8% (spec=78.8%)
PASS: TransactionNameViolation.Hit Details (DOB) missing=62.2% (spec=62.2%)
PASS: TransactionNameViolation.Alerted Party POB missing=100.0% (spec=100.0%)
PASS: Rule.Beneficiary Id Number missing=97.1% (spec=97.1%)
PASS: Rule.Beneficiary Name missing=22.9% (spec=22.9%)
PASS: Rule.Currency Name missing=16.9% (spec=16.9%)
PASS: Rule.Purpose missing=14.8% (spec=14.8%)


## 5. Label / outcome audit

**This is the most important section.** The workbook's operational status fields (`Released`, `UPS`, `Followup`, `Status`) are not trustworthy ground-truth labels. Evidence: how many `UPS`-status rows literally contain "false positive" language in the post-review comment.

In [8]:
def false_positive_flag(series):
    return series.fillna("").str.lower().str.contains("false positive")

status_report = {}

for name, status_col, comment_col in [
    ("CustomerViolation", "Alert Status", "Maker Comment"),
    ("TransactionNameViolation", "Alert Status", "Maker Comment"),
    ("Rule", "Status", "Comment"),
]:
    df = sheets[name]
    dist = df[status_col].value_counts(dropna=False).to_dict()
    ups_mask = df[status_col].astype(str).str.strip().str.upper() == "UPS"
    fp_mask = false_positive_flag(df[comment_col])
    ups_with_fp = int((ups_mask & fp_mask).sum())
    n_ups = int(ups_mask.sum())

    status_report[name] = {
        "status_distribution": {str(k): int(v) for k, v in dist.items()},
        "n_ups": n_ups,
        "ups_rows_containing_false_positive_comment": ups_with_fp,
        "total_rows_containing_false_positive_comment": int(fp_mask.sum()),
    }
    print(f"=== {name} ===")
    print("status distribution:", dist)
    print(f"UPS rows: {n_ups}, of which comment contains 'false positive': {ups_with_fp}")
    print(f"Total rows with 'false positive' in comment: {int(fp_mask.sum())} ({fp_mask.mean()*100:.1f}%)")
    print()

print("CONCLUSION: UPS cannot be mapped to a positive (true-match) label -- a materially")
print("large share of UPS-status rows are themselves described as false positives in the")
print("post-review comment. Status fields require UNKNOWN/RESOLVED/CONFIRMED semantics,")
print("not a manufactured TRUE/FALSE target. See master plan section 5 and Appendix B rule 1.")

=== CustomerViolation ===
status distribution: {'Released': 2204, 'UPS': 193}
UPS rows: 193, of which comment contains 'false positive': 185
Total rows with 'false positive' in comment: 2155 (89.9%)

=== TransactionNameViolation ===
status distribution: {'Released': 1970, 'UPS': 28, nan: 2}
UPS rows: 28, of which comment contains 'false positive': 28
Total rows with 'false positive' in comment: 1881 (94.0%)

=== Rule ===
status distribution: {'Released': 1983, 'UPS': 260, 'Followup': 1}
UPS rows: 260, of which comment contains 'false positive': 1
Total rows with 'false positive' in comment: 74 (3.3%)

CONCLUSION: UPS cannot be mapped to a positive (true-match) label -- a materially
large share of UPS-status rows are themselves described as false positives in the
post-review comment. Status fields require UNKNOWN/RESOLVED/CONFIRMED semantics,
not a manufactured TRUE/FALSE target. See master plan section 5 and Appendix B rule 1.


### Recommended outcome taxonomy (documentation only -- not applied to data in this phase)

| Internal outcome | Meaning | Use initially |
|---|---|---|
| `UNKNOWN` | No trustworthy outcome established | Do not use as supervised target |
| `HISTORICALLY_RELEASED` | Operationally released | Historical context only, not proof of false positive |
| `CONFIRMED_FALSE_POSITIVE` | Explicitly confirmed by reliable evidence | Candidate supervised negative label (future) |
| `CONFIRMED_TRUE_MATCH` | Confirmed genuine match | Candidate supervised positive label (future) |
| `ESCALATED` / `FOLLOW_UP` | Requires additional review | Keep separate from positive/negative |

No row in the current sample is mapped to `CONFIRMED_FALSE_POSITIVE` / `CONFIRMED_TRUE_MATCH` yet -- that mapping requires independently validated evidence the workbook does not provide on its own.

## 6. Leakage registry

Fields that only exist *after* a human reviewed the alert. These must never be used as live inference features -- doing so would let the model see its own answer (master plan section 1 rule, section 8 forbidden-features list).

In [9]:
LEAKAGE_REGISTRY = {
    "CustomerViolation": [
        "Maker Name", "Maker Comment Date", "Maker Comment", "Alert Closure Date & Time",
    ],
    "TransactionNameViolation": [
        "Maker Name", "Maker Comment Date", "Maker Comment",
    ],
    "Rule": [
        "Comment", "Actiondate", "Action Taken By",
    ],
}

for name, cols in LEAKAGE_REGISTRY.items():
    present = [c for c in cols if c in sheets[name].columns]
    missing_from_schema = [c for c in cols if c not in sheets[name].columns]
    print(f"{name}: leakage fields present -> {present}")
    if missing_from_schema:
        print(f"  WARNING: expected leakage field(s) not found in schema: {missing_from_schema}")

with open(REPO_ROOT / "pipelines" / "ingestion" / "leakage_registry.json", "w") as f:
    json.dump(LEAKAGE_REGISTRY, f, indent=2)
print("\nPersisted to pipelines/ingestion/leakage_registry.json for reuse by later phases.")

CustomerViolation: leakage fields present -> ['Maker Name', 'Maker Comment Date', 'Maker Comment', 'Alert Closure Date & Time']
TransactionNameViolation: leakage fields present -> ['Maker Name', 'Maker Comment Date', 'Maker Comment']
Rule: leakage fields present -> ['Comment', 'Actiondate', 'Action Taken By']

Persisted to pipelines/ingestion/leakage_registry.json for reuse by later phases.


## 7. Temporal integrity

Checks whether recorded timestamps are internally consistent. Anomalies here are *not* silently corrected -- they must be clarified with the client before any time-based feature is built (master plan section 3 data-facts table, last row).

In [10]:
temporal_report = {}

# TransactionNameViolation: alert generated before the transaction it concerns?
df = sheets["TransactionNameViolation"]
mask = df["Alert Generated Date & Time"].notna() & df["Trxn date & Time"].notna()
early = (df.loc[mask, "Alert Generated Date & Time"] < df.loc[mask, "Trxn date & Time"]).sum()
temporal_report["TransactionNameViolation_alert_before_trxn"] = {
    "count": int(early), "of": int(mask.sum()), "pct": round(early / mask.sum() * 100, 1)
}
print(f"TransactionNameViolation: alert generated earlier than trxn date: {early}/{mask.sum()} "
      f"({early/mask.sum()*100:.1f}%)")

# Rule: scan date before the transaction date?
df = sheets["Rule"]
mask = df["Scan Date"].notna() & df["Transaction Date"].notna()
early = (df.loc[mask, "Scan Date"] < df.loc[mask, "Transaction Date"]).sum()
temporal_report["Rule_scan_before_trxn"] = {
    "count": int(early), "of": int(mask.sum()), "pct": round(early / mask.sum() * 100, 1)
}
print(f"Rule: scan date earlier than transaction date: {early}/{mask.sum()} ({early/mask.sum()*100:.1f}%)")

# CustomerViolation: closure date before alert generation?
df = sheets["CustomerViolation"]
mask = df["Alert Closure Date & Time"].notna() & df["Alert Generated Date & Time"].notna()
early = (df.loc[mask, "Alert Closure Date & Time"] < df.loc[mask, "Alert Generated Date & Time"]).sum()
temporal_report["CustomerViolation_closure_before_alert"] = {
    "count": int(early), "of": int(mask.sum()), "pct": round(early / mask.sum() * 100, 1) if mask.sum() else 0.0
}
print(f"CustomerViolation: closure earlier than alert generation: {early}/{mask.sum()} "
      f"({early/mask.sum()*100 if mask.sum() else 0:.1f}%)")

print("\nThese are reported, not corrected. Business timestamp semantics must be confirmed")
print("with the client before any temporal feature depends on them (master plan rule).")

TransactionNameViolation: alert generated earlier than trxn date: 1605/2000 (80.2%)
Rule: scan date earlier than transaction date: 328/2244 (14.6%)
CustomerViolation: closure earlier than alert generation: 3/2397 (0.1%)

These are reported, not corrected. Business timestamp semantics must be confirmed
with the client before any temporal feature depends on them (master plan rule).


## 8. Phase 1 summary artifact

Persist a structured PASS/WARN report other phases (and the human approval gate) can reference, per master plan section 21 output-format requirement.

In [11]:
summary = {
    "phase": "1_data_audit",
    "status": "PASS",
    "dataset_version": "v1",
    "raw_data_path": str(RAW_PATH),
    "schema_check": schema_check,
    "duplicate_rows": dup_report,
    "unique_identifier_counts": id_report,
    "missingness_pct_by_field": missingness,
    "label_outcome_audit": status_report,
    "leakage_registry": LEAKAGE_REGISTRY,
    "temporal_integrity": temporal_report,
    "warnings": [
        "POB is 100% missing in both name-alert sheets -- excluded from all feature work "
        "until the client provides source data.",
        "Rule sheet has no clean structured transaction amount field -- do not depend on "
        "comment-parsed amounts in production.",
        "TransactionNameViolation alert-generated timestamp precedes the transaction "
        "timestamp in 80.2% of rows, and Rule scan date precedes transaction date in 14.6% "
        "of rows -- business semantics must be confirmed with the client before building "
        "any time-based feature on these fields.",
        "UPS status cannot be treated as a true-match label -- comment text contradicts it "
        "in a majority of UPS rows for two of the three sheets.",
    ],
    "next_gate": "Phase 2 -- data pipeline / normalization. Requires human approval before proceeding.",
}

out_dir = REPO_ROOT / "evaluation"
out_dir.mkdir(exist_ok=True)

with open(out_dir / "phase1_data_audit_report.json", "w") as f:
    json.dump(summary, f, indent=2, default=str)

print(f"Report written to {out_dir / 'phase1_data_audit_report.json'}")
print(json.dumps(summary, indent=2, default=str)[:2000], "...")

Report written to /home/chpl/Documents/AI-validation/AI_validator/Alert-AI/evaluation/phase1_data_audit_report.json
{
  "phase": "1_data_audit",
  "status": "PASS",
  "dataset_version": "v1",
  "raw_data_path": "/home/chpl/Documents/AI-validation/AI_validator/Alert-AI/data/raw/Alerts_Samples.xlsx",
  "schema_check": {
    "CustomerViolation": {
      "rows": 2397,
      "cols": 23,
      "expected_rows": 2397,
      "expected_cols": 23,
      "match": true
    },
    "TransactionNameViolation": {
      "rows": 2000,
      "cols": 25,
      "expected_rows": 2000,
      "expected_cols": 25,
      "match": true
    },
    "Rule": {
      "rows": 2244,
      "cols": 24,
      "expected_rows": 2244,
      "expected_cols": 24,
      "match": true
    }
  },
  "duplicate_rows": {
    "CustomerViolation": 8,
    "TransactionNameViolation": 537,
    "Rule": 0
  },
  "unique_identifier_counts": {
    "CustomerViolation": {
      "UIN": 1232
    },
    "TransactionNameViolation": {
      "UIN": 3

## Phase 1 -- Result

**Status: PASS**

- Schema, row/column counts confirmed to match the audited spec exactly.
- Duplicate rows identified and quantified (not removed -- deduplication is a Phase 2 decision).
- Repeated-entity structure (UIN, transaction reference, customer number) confirmed present -- required for group-based validation in later phases.
- Missingness confirmed field-by-field against the spec's specific claims.
- Label audit confirms `UPS`/`Released` are not usable as supervised ground truth; outcome taxonomy documented for future use, not applied.
- Leakage registry persisted to `pipelines/ingestion/leakage_registry.json` for reuse by the feature-engineering phases.
- Temporal anomalies reported, not silently corrected.

**Next gate:** Phase 2 -- data pipeline (ingestion, validation, normalization, dataset versioning). Awaiting human approval to proceed.